# Large Synthetic Conformational Graph Experiment

This notebook runs the recursive Gaussian synthetic experiment through the shared package code in `guided_conformational_path.large_synthetic`.

The main run cell displays both a `tqdm` progress bar and an HTML progress bar so progress is visible in Jupyter, VS Code notebooks, and fallback renderers.

In [ ]:
from pathlib import Path

from IPython.display import HTML, Image, display
from tqdm.auto import tqdm

from guided_conformational_path.large_synthetic import (
    LargeSyntheticConfig,
    PDBEndpointConfig,
    display_py3dmol_path,
    format_run_summary,
    route_family_costs,
    run_large_synthetic_experiment,
    run_pdb_endpoint_experiment,
    write_run_outputs,
)


def _progress_html(desc, current, total, event=None, complete=False):
    percent = 0.0 if total == 0 else min(100.0, 100.0 * current / total)
    status = "complete" if complete else "running"
    detail = "waiting for first iteration"
    if event:
        detail = (
            f"iteration={event['iteration']}/{event['max_iterations']} | "
            f"accepted={event['accepted_this_iteration']} | "
            f"generated={event['returned_total']}/{event['requested_total']} | "
            f"frontier=A{event['frontier_a']}/B{event['frontier_b']}"
        )
    return f"""
    <div style='font-family: ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; margin: 10px 0 14px 0;'>
      <div style='font-weight: 700; margin-bottom: 5px;'>{desc}: {current}/{total} iterations ({percent:0.1f}%)</div>
      <div style='width: 100%; max-width: 760px; height: 20px; border-radius: 999px; background: #e5e7eb; overflow: hidden; border: 1px solid #d1d5db;'>
        <div style='width: {percent:0.3f}%; height: 100%; background: linear-gradient(90deg, #2563eb, #dc2626); transition: width 120ms linear;'></div>
      </div>
      <div style='font-size: 12px; color: #374151; margin-top: 5px;'>{status} | {detail}</div>
    </div>
    """


def make_iteration_progress(total, desc):
    """Create a tqdm progress bar plus a visible HTML progress display."""
    events = []
    current = 0
    html_handle = display(HTML(_progress_html(desc, current, total)), display_id=True)
    pbar = tqdm(total=total, desc=desc, unit="iter", leave=True, dynamic_ncols=True)

    def update(event):
        nonlocal current
        events.append(event)
        current = min(total, current + 1)
        pbar.update(1)
        pbar.set_postfix(
            accepted=event["accepted_this_iteration"],
            generated=f"{event['returned_total']}/{event['requested_total']}",
            frontier=f"A{event['frontier_a']}/B{event['frontier_b']}",
        )
        pbar.refresh()
        html_handle.update(HTML(_progress_html(desc, current, total, event=event)))

    def close():
        pbar.close()
        last_event = events[-1] if events else None
        html_handle.update(HTML(_progress_html(desc, current, total, event=last_event, complete=True)))

    return update, close, events

## Synthetic Experiment Parameters

Change these values to control graph size and recursive Gaussian sampling.

Set `allow_fallback=False` to inspect the best recursive Gaussian path directly. Set it to `True` if you want the designed waypoint fallback when recursive sampling misses the target path shape.

In [ ]:
config = LargeSyntheticConfig(
    n_residues=200,
    n_samples=80,
    noise_scale=0.15,
    seed=7,
    recursive_max_iterations=6,
    recursive_top_k=8,
    recursive_samples_per_parent=4,
    target_min_intermediates=1,
    target_max_intermediates=100,
    allow_fallback=False,
    fallback_waypoints=4,
    fallback_samples_per_waypoint=0,
    fallback_noise_scale=0.05,
    k_neighbors=0,
    outputs_dir=Path("outputs/notebook_large_synthetic"),
    write_nodes=False,
    write_pymol=True,
    write_py3dmol=True,
    write_animation=True,
    animation_frames_per_edge=20,
    animation_fps=10,
    animation_dpi=120,
    graph_min_edge_weight=None,
    graph_max_edge_weight=None,
    graph_non_path_edge_alpha=0.125,
)
config

## Run Synthetic Experiment

Run this cell to execute the experiment. You should see progress immediately below the cell: first a visible HTML status bar, then tqdm output. The progress updates once after each recursive Gaussian expansion iteration.

In [ ]:
synthetic_progress_callback, synthetic_progress_close, synthetic_progress_events = make_iteration_progress(
    total=config.recursive_max_iterations,
    desc="Recursive Gaussian iterations",
)

try:
    run = run_large_synthetic_experiment(
        config,
        progress_callback=synthetic_progress_callback,
    )
finally:
    synthetic_progress_close()

write_run_outputs(run, config)
print(format_run_summary(run, config.outputs_dir))

## Inspect Synthetic Outputs

In [ ]:
graph_path = config.outputs_dir / "graph.png"
if graph_path.exists():
    display(Image(filename=str(graph_path)))
else:
    print(f"Graph image not found yet: {graph_path}")

run.summary

In [ ]:
# Optional inline 3D path viewer. Requires notebook extras including py3Dmol.
try:
    display_py3dmol_path(run)
except Exception as exc:
    print(f"py3Dmol viewer unavailable: {exc}")

## PDB Endpoint Experiment

Place endpoint files here before running this section:

```text
data/pdb_inputs/A.pdb
data/pdb_inputs/B.pdb
```

This uses the same recursive Gaussian workflow, but starts from user-provided PDB endpoint structures.

In [ ]:
endpoint_a_file = Path("data/pdb_inputs/A.pdb")
endpoint_b_file = Path("data/pdb_inputs/B.pdb")

print(f"A exists: {endpoint_a_file.exists()} -> {endpoint_a_file.resolve()}")
print(f"B exists: {endpoint_b_file.exists()} -> {endpoint_b_file.resolve()}")

In [ ]:
pdb_config = PDBEndpointConfig(
    endpoint_a_path=endpoint_a_file,
    endpoint_b_path=endpoint_b_file,
    n_samples=80,
    noise_scale=0.15,
    seed=7,
    recursive_max_iterations=6,
    recursive_top_k=8,
    recursive_samples_per_parent=4,
    target_min_intermediates=3,
    target_max_intermediates=4,
    allow_fallback=True,
    fallback_waypoints=4,
    fallback_samples_per_waypoint=0,
    fallback_noise_scale=0.05,
    k_neighbors=0,
    outputs_dir=Path("outputs/pdb_endpoint_experiment"),
    write_nodes=False,
    write_pymol=True,
    write_py3dmol=True,
    write_animation=True,
    graph_min_edge_weight=None,
    graph_max_edge_weight=None,
    graph_non_path_edge_alpha=0.125,
    matching_method="exact_then_geometric",
    min_exact_match_fraction=0.8,
    geometric_match_distance_cutoff=5.0,
    min_geometric_matches=8,
)
pdb_config

In [ ]:
pdb_progress_callback, pdb_progress_close, pdb_progress_events = make_iteration_progress(
    total=pdb_config.recursive_max_iterations,
    desc="PDB recursive Gaussian iterations",
)

try:
    pdb_run = run_pdb_endpoint_experiment(
        pdb_config,
        progress_callback=pdb_progress_callback,
    )
finally:
    pdb_progress_close()

write_run_outputs(pdb_run, pdb_config)
print(format_run_summary(pdb_run, pdb_config.outputs_dir))
pdb_run.summary.get("input", {})

## Parameter Notes

- `n_residues`: synthetic C-alpha count.
- `n_samples`: total accepted recursive Gaussian node cap.
- `recursive_max_iterations`: maximum recursive expansion rounds and progress-bar total.
- `recursive_top_k`: number of closest cross-side frontier parents retained per side.
- `recursive_samples_per_parent`: Gaussian children requested for each selected frontier parent.
- `allow_fallback`: if `False`, return the best recursive Gaussian path directly; if `True`, use designed waypoint fallback when the recursive path misses the target shape.
- `k_neighbors=0`: full graph connectivity. Set a positive value for k-nearest-neighbor graph construction.